# Training Model

Notebook ini menjalankan training model klasifikasi kategori market value. Dataset dibaca dari output preprocessing dan split dilakukan berdasarkan musim.

## 1. Install Dependency

In [1]:
%pip install scikit-learn imbalanced-learn xgboost joblib

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Import Library dan Konfigurasi

In [2]:
from pathlib import Path
import importlib
import json
import subprocess
import sys
import warnings

import numpy as np
import pandas as pd


def ensure_package(import_name, package_name=None):
    package_name = package_name or import_name
    try:
        return importlib.import_module(import_name)
    except ModuleNotFoundError:
        print(f'Menginstall dependency: {package_name}')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', package_name])
        return importlib.import_module(import_name)


ensure_package('sklearn', 'scikit-learn')
ensure_package('imblearn', 'imbalanced-learn')
ensure_package('xgboost')
ensure_package('joblib')

from imblearn.over_sampling import RandomOverSampler
from imblearn.pipeline import Pipeline as ImbPipeline
from joblib import dump
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_recall_fscore_support,
)
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler
from xgboost import XGBClassifier

warnings.filterwarnings('ignore')

PROJECT_ROOT = Path.cwd()
MODEL_DATA_FILE = PROJECT_ROOT / 'data' / 'model' / 'players_model.csv'
FEATURE_LIST_FILE = PROJECT_ROOT / 'data' / 'model' / 'feature_list.json'
MATCHING_FILE = PROJECT_ROOT / 'data' / 'interim' / 'player_matching_result.csv'

OUTPUT_DIR = PROJECT_ROOT / 'data' / 'output'
MODELS_DIR = PROJECT_ROOT / 'models'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

MODEL_METRICS_FILE = OUTPUT_DIR / 'model_metrics.csv'
VALIDATION_METRICS_FILE = OUTPUT_DIR / 'validation_metrics.csv'
TEST_METRICS_FILE = OUTPUT_DIR / 'test_metrics.csv'
CLASSIFICATION_REPORT_FILE = OUTPUT_DIR / 'classification_report.csv'
CONFUSION_MATRIX_FILE = OUTPUT_DIR / 'confusion_matrix.csv'
FEATURE_IMPORTANCE_FILE = OUTPUT_DIR / 'feature_importance.csv'
OVERSAMPLING_SUMMARY_FILE = OUTPUT_DIR / 'oversampling_summary.csv'
BEST_MODEL_SUMMARY_FILE = OUTPUT_DIR / 'best_model_summary.csv'
BEST_MODEL_FILE = MODELS_DIR / 'best_model.pkl'
LABEL_ENCODER_FILE = MODELS_DIR / 'label_encoder.pkl'

TRAIN_BEFORE_OVERSAMPLING_FILE = MODEL_DATA_FILE.parent / 'train_before_oversampling.csv'
TRAIN_AFTER_OVERSAMPLING_FILE = MODEL_DATA_FILE.parent / 'train_after_oversampling.csv'
HIGH_BEFORE_OVERSAMPLING_FILE = MODEL_DATA_FILE.parent / 'train_high_before_oversampling.csv'
HIGH_AFTER_OVERSAMPLING_FILE = MODEL_DATA_FILE.parent / 'train_high_after_oversampling.csv'

RANDOM_STATE = 42
TARGET_COLUMN = 'market_value_category'
TARGET_LABELS = ['Rendah', 'Menengah', 'Tinggi']
HIGH_CLASS_LABEL = 'Tinggi'
REBUILD_OVERSAMPLING_DATA = False

print('Project root:', PROJECT_ROOT)


Project root: c:\KULIAH\SEMESTER 6\BIG DATA\UAS\Tugas_UAS_Big-Data


## 3. Load Dataset dan Validasi Awal

In [3]:
if not MODEL_DATA_FILE.exists() or MODEL_DATA_FILE.stat().st_size == 0:
    raise FileNotFoundError('Dataset model belum tersedia. Jalankan 02_preprocessing.ipynb terlebih dahulu.')
if not FEATURE_LIST_FILE.exists() or FEATURE_LIST_FILE.stat().st_size == 0:
    raise FileNotFoundError('feature_list.json belum tersedia. Jalankan 02_preprocessing.ipynb terlebih dahulu.')

df = pd.read_csv(MODEL_DATA_FILE)
with open(FEATURE_LIST_FILE, 'r', encoding='utf-8') as file:
    feature_list = json.load(file)

missing_features = [col for col in feature_list if col not in df.columns]
if missing_features:
    raise ValueError(f'Fitur tidak tersedia di dataset model: {missing_features}')
if TARGET_COLUMN not in df.columns:
    raise ValueError('Target column tidak tersedia di dataset model.')
if 'season' not in df.columns:
    raise ValueError('Kolom season wajib tersedia untuk time-based split.')
if df.empty:
    raise ValueError('Dataset model kosong.')

forbidden_features = {
    'market_value_mio', 'market_value_str', 'market_value_category',
    'value_category', 'label', 'target', 'player_id', 'player_name',
    'player_url', 'position_detail', 'mv_growth_rate'
}
forbidden_in_features = sorted(set(feature_list).intersection(forbidden_features))
if forbidden_in_features:
    raise ValueError(f'Forbidden feature masuk ke feature_list: {forbidden_in_features}')

print('Rows:', len(df))
print('Features:', len(feature_list))
print(df[TARGET_COLUMN].value_counts())

Rows: 10211
Features: 37
market_value_category
Rendah      5534
Menengah    3395
Tinggi      1282
Name: count, dtype: int64


## 4. Time-Based Split

In [4]:
train_df = df[df['season'].between(2017, 2021)].copy()
validation_df = df[df['season'] == 2022].copy()
test_df = df[df['season'].between(2023, 2024)].copy()

if train_df.empty or validation_df.empty or test_df.empty:
    raise ValueError(
        f'Split kosong. train={len(train_df)}, validation={len(validation_df)}, test={len(test_df)}'
    )

train_before_df = train_df.reset_index(drop=True).copy()
X_train_before = train_before_df[feature_list].copy()
X_validation = validation_df[feature_list].copy()
X_test = test_df[feature_list].copy()

y_train_text_before = train_before_df[TARGET_COLUMN].copy()
y_validation_text = validation_df[TARGET_COLUMN].copy()
y_test_text = test_df[TARGET_COLUMN].copy()

label_encoder = LabelEncoder()
label_encoder.fit(TARGET_LABELS)
y_train_before = label_encoder.transform(y_train_text_before)
y_validation = label_encoder.transform(y_validation_text)
y_test = label_encoder.transform(y_test_text)

print('Train rows sebelum oversampling:', len(train_before_df))
print('Validation rows:', len(validation_df))
print('Test rows:', len(test_df))
print('\nTrain label distribution sebelum oversampling:')
print(y_train_text_before.value_counts())
print('\nValidation label distribution:')
print(y_validation_text.value_counts())
print('\nTest label distribution:')
print(y_test_text.value_counts())


Train rows sebelum oversampling: 6138
Validation rows: 1334
Test rows: 2739

Train label distribution sebelum oversampling:
market_value_category
Rendah      3403
Menengah    1977
Tinggi       758
Name: count, dtype: int64

Validation label distribution:
market_value_category
Rendah      726
Menengah    446
Tinggi      162
Name: count, dtype: int64

Test label distribution:
market_value_category
Rendah      1405
Menengah     972
Tinggi       362
Name: count, dtype: int64


## 5. Preprocessor, Model, dan Sampling Scenario

In [5]:
categorical_features = [col for col in feature_list if X_train_before[col].dtype == 'object']
numeric_features = [col for col in feature_list if col not in categorical_features]


def make_one_hot_encoder():
    try:
        return OneHotEncoder(handle_unknown='ignore', sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown='ignore', sparse=False)


def make_preprocessor(scale_numeric=False):
    numeric_transformer = StandardScaler() if scale_numeric else 'passthrough'
    return ColumnTransformer(
        transformers=[
            ('numeric', numeric_transformer, numeric_features),
            ('categorical', make_one_hot_encoder(), categorical_features),
        ],
        remainder='drop',
        verbose_feature_names_out=False,
    )


def light_oversampling_strategy(y):
    counts = pd.Series(y).value_counts().to_dict()
    majority_count = max(counts.values())
    target_count = int(majority_count * 0.85)
    strategy = {}
    for label_value, count in counts.items():
        if count < target_count:
            strategy[label_value] = target_count
    return strategy


def cache_is_fresh(path):
    if not path.exists() or path.stat().st_size == 0:
        return False
    latest_input_mtime = max(MODEL_DATA_FILE.stat().st_mtime, FEATURE_LIST_FILE.stat().st_mtime)
    return path.stat().st_mtime >= latest_input_mtime


def build_oversampling_summary(before_df, after_df):
    before_counts = before_df[TARGET_COLUMN].value_counts().reindex(TARGET_LABELS, fill_value=0).astype(int)
    after_counts = after_df[TARGET_COLUMN].value_counts().reindex(TARGET_LABELS, fill_value=0).astype(int)
    summary = pd.DataFrame({
        'category': TARGET_LABELS,
        'before_records': [int(before_counts[label]) for label in TARGET_LABELS],
        'after_records': [int(after_counts[label]) for label in TARGET_LABELS],
    })
    summary['added_records'] = summary['after_records'] - summary['before_records']
    summary['oversampling_ratio_to_majority'] = 0.85
    summary['source_split'] = 'train_2017_2021'
    return summary


def prepare_oversampled_train_dataset():
    train_before_for_export = train_before_df.copy().reset_index(drop=True)
    train_before_for_export['_source_row_id'] = np.arange(len(train_before_for_export))
    train_before_for_export['_oversampling_status'] = 'Original'
    train_before_for_export['_is_oversampled_duplicate'] = False
    train_before_for_export.to_csv(TRAIN_BEFORE_OVERSAMPLING_FILE, index=False)
    train_before_for_export[train_before_for_export[TARGET_COLUMN] == HIGH_CLASS_LABEL].to_csv(
        HIGH_BEFORE_OVERSAMPLING_FILE, index=False
    )

    required_cache_columns = list(train_before_for_export.columns)
    use_cache = cache_is_fresh(TRAIN_AFTER_OVERSAMPLING_FILE) and not REBUILD_OVERSAMPLING_DATA
    if use_cache:
        train_after_df = pd.read_csv(TRAIN_AFTER_OVERSAMPLING_FILE)
        missing_columns = [col for col in required_cache_columns if col not in train_after_df.columns]
        if missing_columns:
            print(f'Cache oversampling tidak lengkap, dibuat ulang. Kolom hilang: {missing_columns}')
            use_cache = False
        else:
            print(f'Pakai cache oversampling: {TRAIN_AFTER_OVERSAMPLING_FILE}')

    if not use_cache:
        sampler = RandomOverSampler(
            sampling_strategy=light_oversampling_strategy(y_train_text_before),
            random_state=RANDOM_STATE,
        )
        train_after_df, _ = sampler.fit_resample(train_before_for_export, y_train_text_before)
        train_after_df = train_after_df.reset_index(drop=True)
        duplicate_mask = np.arange(len(train_after_df)) >= len(train_before_for_export)
        train_after_df['_is_oversampled_duplicate'] = duplicate_mask
        train_after_df['_oversampling_status'] = np.where(
            duplicate_mask, 'Tambahan oversampling', 'Original'
        )
        train_after_df.to_csv(TRAIN_AFTER_OVERSAMPLING_FILE, index=False)
        print(f'Oversampling dibuat dan disimpan: {TRAIN_AFTER_OVERSAMPLING_FILE}')

    train_after_df[train_after_df[TARGET_COLUMN] == HIGH_CLASS_LABEL].to_csv(
        HIGH_AFTER_OVERSAMPLING_FILE, index=False
    )
    oversampling_summary_df = build_oversampling_summary(train_before_for_export, train_after_df)
    oversampling_summary_df.to_csv(OVERSAMPLING_SUMMARY_FILE, index=False)
    return train_after_df, oversampling_summary_df


train_after_oversampling_df, oversampling_summary_df = prepare_oversampled_train_dataset()

X_train = train_after_oversampling_df[feature_list].copy()
y_train_text = train_after_oversampling_df[TARGET_COLUMN].copy()
y_train = label_encoder.transform(y_train_text)


def get_training_data(scenario):
    if scenario == 'random_oversampling_light':
        return X_train, y_train
    return X_train_before, y_train_before


def build_pipeline(model_name, scenario):
    scale_numeric = model_name == 'logistic_regression'
    steps = [('preprocess', make_preprocessor(scale_numeric=scale_numeric))]

    class_weight = 'balanced' if scenario == 'class_weight_balanced' else None

    if model_name == 'logistic_regression':
        model = LogisticRegression(
            max_iter=2000,
            class_weight=class_weight,
            random_state=RANDOM_STATE,
            n_jobs=-1,
        )
    elif model_name == 'xgboost':
        model = XGBClassifier(
            n_estimators=700,
            max_depth=4,
            learning_rate=0.04,
            subsample=0.9,
            colsample_bytree=0.9,
            objective='multi:softprob',
            eval_metric='mlogloss',
            random_state=RANDOM_STATE,
            n_jobs=-1,
        )
    else:
        model = HistGradientBoostingClassifier(random_state=RANDOM_STATE)

    steps.append(('model', model))
    return ImbPipeline(steps)


model_names = ['logistic_regression', 'xgboost']
scenarios_by_model = {
    'logistic_regression': ['no_sampling', 'class_weight_balanced', 'random_oversampling_light'],
    'xgboost': ['no_sampling', 'random_oversampling_light'],
}

print('Numeric features:', len(numeric_features))
print('Categorical features:', categorical_features)
print('\nOversampling strategy:', light_oversampling_strategy(y_train_text_before))
print('\nOversampling summary:')
display(oversampling_summary_df)


Oversampling dibuat dan disimpan: c:\KULIAH\SEMESTER 6\BIG DATA\UAS\Tugas_UAS_Big-Data\data\model\train_after_oversampling.csv
Numeric features: 32
Categorical features: ['preferred_foot', 'pos_category', 'league', 'prev_mv_category', 'two_seasons_ago_mv_category']

Oversampling strategy: {'Menengah': 2892, 'Tinggi': 2892}

Oversampling summary:


,category,before_records,after_records,added_records,oversampling_ratio_to_majority,source_split
0,Rendah,3403,3403,0,0.85,train_2017_2021
1,Menengah,1977,2892,915,0.85,train_2017_2021
2,Tinggi,758,2892,2134,0.85,train_2017_2021


## 6. Helper Evaluasi

In [6]:
def evaluate_predictions(y_true, y_pred):
    target_names = list(label_encoder.inverse_transform([0, 1, 2]))
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, labels=[0, 1, 2], zero_division=0
    )
    macro_precision, macro_recall, macro_f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average='macro', zero_division=0
    )
    weighted_f1 = precision_recall_fscore_support(
        y_true, y_pred, average='weighted', zero_division=0
    )[2]

    metrics = {
        'accuracy': accuracy_score(y_true, y_pred),
        'macro_precision': macro_precision,
        'macro_recall': macro_recall,
        'macro_f1': macro_f1,
        'weighted_f1': weighted_f1,
    }
    for index, label_name in enumerate(target_names):
        key = label_name.lower()
        metrics[f'recall_{key}'] = recall[index]
        metrics[f'precision_{key}'] = precision[index]
        metrics[f'f1_{key}'] = f1[index]
    return metrics


def build_metric_row(model_name, scenario, split_name, metrics):
    row = {'model': model_name, 'scenario': scenario, 'split': split_name}
    row.update(metrics)
    return row


def get_transformed_feature_names(pipeline):
    preprocessor = pipeline.named_steps['preprocess']
    try:
        return list(preprocessor.get_feature_names_out())
    except Exception:
        return [f'feature_{i}' for i in range(pipeline.named_steps['model'].n_features_in_)]


def extract_feature_importance(pipeline, model_name):
    model = pipeline.named_steps['model']
    names = get_transformed_feature_names(pipeline)
    if hasattr(model, 'feature_importances_'):
        importance = model.feature_importances_
    elif hasattr(model, 'coef_'):
        importance = np.abs(model.coef_).mean(axis=0)
    else:
        importance = np.zeros(len(names))
    result = pd.DataFrame({'feature': names, 'importance': importance})
    result['model'] = model_name
    result = result.sort_values('importance', ascending=False).reset_index(drop=True)
    return result


## 7. Training Kandidat dan Seleksi Validation

In [7]:
validation_metric_rows = []
trained_candidates = []

for model_name in model_names:
    for scenario in scenarios_by_model[model_name]:
        print(f'Training {model_name} | {scenario}')
        pipeline = build_pipeline(model_name, scenario)
        scenario_X_train, scenario_y_train = get_training_data(scenario)
        pipeline.fit(scenario_X_train, scenario_y_train)
        validation_pred = pipeline.predict(X_validation)
        validation_metrics = evaluate_predictions(y_validation, validation_pred)
        validation_metric_rows.append(build_metric_row(model_name, scenario, 'validation', validation_metrics))
        trained_candidates.append({
            'model_name': model_name,
            'scenario': scenario,
            'pipeline': pipeline,
            'validation_metrics': validation_metrics,
            'train_rows_used': len(scenario_X_train),
        })

validation_metrics_df = pd.DataFrame(validation_metric_rows)
validation_metrics_df = validation_metrics_df.sort_values(
    ['macro_f1', 'accuracy'], ascending=[False, False]
).reset_index(drop=True)
display(validation_metrics_df[['model', 'scenario', 'accuracy', 'macro_f1', 'weighted_f1']])

best_model_name = validation_metrics_df.loc[0, 'model']
best_scenario = validation_metrics_df.loc[0, 'scenario']
best_candidate = next(
    candidate for candidate in trained_candidates
    if candidate['model_name'] == best_model_name and candidate['scenario'] == best_scenario
)
best_pipeline = best_candidate['pipeline']

print('Best model:', best_model_name)
print('Best scenario:', best_scenario)
print('Train rows used by best scenario:', best_candidate['train_rows_used'])


Training logistic_regression | no_sampling
Training logistic_regression | class_weight_balanced
Training logistic_regression | random_oversampling_light
Training xgboost | no_sampling
Training xgboost | random_oversampling_light


,model,scenario,accuracy,macro_f1,weighted_f1
0,xgboost,random_oversampling_light,0.817841,0.788557,0.816564
1,xgboost,no_sampling,0.820840,0.788121,0.817905
2,logistic_regression,no_sampling,0.809595,0.776452,0.804756
3,logistic_regression,random_oversampling_light,0.799100,0.774101,0.798560
4,logistic_regression,class_weight_balanced,0.795352,0.773620,0.796274


Best model: xgboost
Best scenario: random_oversampling_light
Train rows used by best scenario: 9187


## 8. Evaluasi Final pada Test Set

In [8]:
validation_pred = best_pipeline.predict(X_validation)
test_pred = best_pipeline.predict(X_test)

best_validation_metrics = evaluate_predictions(y_validation, validation_pred)
test_metrics = evaluate_predictions(y_test, test_pred)

test_metric_row = build_metric_row(best_model_name, best_scenario, 'test', test_metrics)
validation_metric_row = build_metric_row(best_model_name, best_scenario, 'validation', best_validation_metrics)
model_metrics_df = pd.concat([
    validation_metrics_df,
    pd.DataFrame([test_metric_row]),
], ignore_index=True)

target_names = list(label_encoder.inverse_transform([0, 1, 2]))
classification_report_df = pd.DataFrame(
    classification_report(
        y_test,
        test_pred,
        labels=[0, 1, 2],
        target_names=target_names,
        zero_division=0,
        output_dict=True,
    )
).transpose().reset_index().rename(columns={'index': 'label'})

confusion_df = pd.DataFrame(
    confusion_matrix(y_test, test_pred, labels=[0, 1, 2]),
    index=[f'actual_{label}' for label in target_names],
    columns=[f'predicted_{label}' for label in target_names],
).reset_index().rename(columns={'index': 'actual_label'})

feature_importance_df = extract_feature_importance(best_pipeline, best_model_name)

best_model_summary_df = pd.DataFrame([{
    'model': best_model_name,
    'scenario': best_scenario,
    'target_column': TARGET_COLUMN,
    'prediction_type': 'classification',
    'label_order': '|'.join(TARGET_LABELS),
    'value_range_rendah': '5 <= value < 15 EUR juta',
    'value_range_menengah': '15 <= value <= 35 EUR juta',
    'value_range_tinggi': '> 35 EUR juta',
    'validation_accuracy': best_validation_metrics['accuracy'],
    'validation_macro_f1': best_validation_metrics['macro_f1'],
    'test_accuracy': test_metrics['accuracy'],
    'test_macro_f1': test_metrics['macro_f1'],
    'train_rows_before_oversampling': len(train_before_df),
    'train_rows_after_oversampling': len(train_after_oversampling_df),
}])

display(pd.DataFrame([validation_metric_row, test_metric_row])[['split', 'accuracy', 'macro_f1', 'weighted_f1']])
display(confusion_df)
display(feature_importance_df.head(20))


,split,accuracy,macro_f1,weighted_f1
0,validation,0.817841,0.788557,0.816564
1,test,0.815991,0.798896,0.813489


,actual_label,predicted_Menengah,predicted_Rendah,predicted_Tinggi
0,actual_Menengah,683,230,59
1,actual_Rendah,127,1278,0
2,actual_Tinggi,85,3,274


,feature,importance,model
0,prev_mv_category_Menengah,0.227210,xgboost
1,prev_season_mv,0.090992,xgboost
2,prev_season_mv_log,0.088002,xgboost
3,prev_mv_category_Tinggi,0.078914,xgboost
4,prev_mv_category_Rendah,0.029098,xgboost
5,has_prev_mv,0.027419,xgboost
6,club_total_mv_mio,0.026350,xgboost
7,shots_total,0.024502,xgboost
8,club_mv_relative_to_league_avg,0.019309,xgboost
9,mv_history_count,0.018695,xgboost


## 9. Simpan Artifact

In [9]:
model_metrics_df.to_csv(MODEL_METRICS_FILE, index=False)
pd.DataFrame([validation_metric_row]).to_csv(VALIDATION_METRICS_FILE, index=False)
pd.DataFrame([test_metric_row]).to_csv(TEST_METRICS_FILE, index=False)
classification_report_df.to_csv(CLASSIFICATION_REPORT_FILE, index=False)
confusion_df.to_csv(CONFUSION_MATRIX_FILE, index=False)
feature_importance_df.to_csv(FEATURE_IMPORTANCE_FILE, index=False)
oversampling_summary_df.to_csv(OVERSAMPLING_SUMMARY_FILE, index=False)
best_model_summary_df.to_csv(BEST_MODEL_SUMMARY_FILE, index=False)
dump(best_pipeline, BEST_MODEL_FILE)
dump(label_encoder, LABEL_ENCODER_FILE)

required_outputs = [
    MODEL_METRICS_FILE, VALIDATION_METRICS_FILE, TEST_METRICS_FILE,
    CLASSIFICATION_REPORT_FILE, CONFUSION_MATRIX_FILE, FEATURE_IMPORTANCE_FILE,
    OVERSAMPLING_SUMMARY_FILE, BEST_MODEL_SUMMARY_FILE, TRAIN_BEFORE_OVERSAMPLING_FILE, TRAIN_AFTER_OVERSAMPLING_FILE,
    HIGH_BEFORE_OVERSAMPLING_FILE, HIGH_AFTER_OVERSAMPLING_FILE,
    BEST_MODEL_FILE, LABEL_ENCODER_FILE,
]
missing_outputs = [str(path) for path in required_outputs if not path.exists() or path.stat().st_size == 0]
if missing_outputs:
    raise ValueError(f'Output training gagal dibuat: {missing_outputs}')


## 10. Final Report

In [10]:
if MATCHING_FILE.exists() and MATCHING_FILE.stat().st_size > 0:
    matching_df = pd.read_csv(MATCHING_FILE)
    fbref_matched_rows = int(matching_df['matched'].sum())
    fbref_unmatched_rows = int((~matching_df['matched']).sum())
else:
    fbref_matched_rows = 0
    fbref_unmatched_rows = 0

print('Training selesai.')
print(f'Best model              : {best_model_name}')
print(f'Best scenario           : {best_scenario}')
print(f'Validation macro F1     : {best_validation_metrics["macro_f1"]:.4f}')
print(f'Validation accuracy     : {best_validation_metrics["accuracy"]:.4f}')
print(f'Test macro F1           : {test_metrics["macro_f1"]:.4f}')
print(f'Test accuracy           : {test_metrics["accuracy"]:.4f}')
print(f'Train rows before OS    : {len(train_before_df)}')
print(f'Train rows after OS     : {len(train_after_oversampling_df)}')
print(f'Validation rows         : {len(validation_df)}')
print(f'Test rows               : {len(test_df)}')
print(f'FBref matched rows      : {fbref_matched_rows}')
print(f'FBref unmatched rows    : {fbref_unmatched_rows}')
print(f'Output metrics          : {MODEL_METRICS_FILE.relative_to(PROJECT_ROOT)}')
print(f'Oversampling summary    : {OVERSAMPLING_SUMMARY_FILE.relative_to(PROJECT_ROOT)}')
print(f'Best model summary      : {BEST_MODEL_SUMMARY_FILE.relative_to(PROJECT_ROOT)}')
print(f'Train before OS         : {TRAIN_BEFORE_OVERSAMPLING_FILE.relative_to(PROJECT_ROOT)}')
print(f'Train after OS          : {TRAIN_AFTER_OVERSAMPLING_FILE.relative_to(PROJECT_ROOT)}')
print(f'High before OS          : {HIGH_BEFORE_OVERSAMPLING_FILE.relative_to(PROJECT_ROOT)}')
print(f'High after OS           : {HIGH_AFTER_OVERSAMPLING_FILE.relative_to(PROJECT_ROOT)}')
print(f'Best model artifact     : {BEST_MODEL_FILE.relative_to(PROJECT_ROOT)}')


Training selesai.
Best model              : xgboost
Best scenario           : random_oversampling_light
Validation macro F1     : 0.7886
Validation accuracy     : 0.8178
Test macro F1           : 0.7989
Test accuracy           : 0.8160
Train rows before OS    : 6138
Train rows after OS     : 9187
Validation rows         : 1334
Test rows               : 2739
FBref matched rows      : 9836
FBref unmatched rows    : 375
Output metrics          : data\output\model_metrics.csv
Oversampling summary    : data\output\oversampling_summary.csv
Best model summary      : data\output\best_model_summary.csv
Train before OS         : data\model\train_before_oversampling.csv
Train after OS          : data\model\train_after_oversampling.csv
High before OS          : data\model\train_high_before_oversampling.csv
High after OS           : data\model\train_high_after_oversampling.csv
Best model artifact     : models\best_model.pkl
